## Milestone 5: Statistical Analysis

All tests use a 5% significance level. Daily totals are used wherever transactions onthe same day would not be independent.

| # | Question | Test | Result |
|---|---|---|---|
| 1 | Did promotions raise sales? (Success Metrics: promotional lift) | Welch t-test + 95% CI on daily totals (213 promo vs 456 other days) | Net sales -8,331 KES/day (CI -10,200 to -6,462, p < 0.001); gross profit -7,982 KES/day (CI -8,440 to -7,524, p < 0.001); transactions no difference (CI -3 to +3, p = 0.98) |
| 2 | Q4: Do holidays change sales? | Mann-Whitney U vs ordinary days (22 holidays, 57 pre-holiday days, 590 ordinary days) | Holiday net sales -4.2% (p = 0.54); 3 days before +0.2% (p = 0.94); basket value -1.7% (p = 0.38) and -2.5% (p = 0.12). None significant |
| 3 | Q4: Does inflation move sales? | Spearman, 21 months | Net sales rho = -0.12 (p = 0.62); avg daily sales -0.10 (p = 0.68); basket -0.12 (p = 0.60). None significant |
| 4 | Q4: Do sales scale with county population? | Spearman across 6 counties, exact permutation p-value | rho = 0.83, exact p = 0.058, not significant |
| 5 | Q1: Does payment mix differ by county or category? | Chi-square + Cramér's V (Unknown payment excluded) | County: p = 0.19, V = 0.011. Category: p = 0.73, V = 0.006. Not significant |

**Findings**

**Promotions did not work.**

They brought no extra customers, so the discount only reduced revenue (about 14%) and gross profit (about 50%) per day.

**Holidays and inflation show no detectable effect**

On daily sales, basket value or monthly sales.

**County population:** 

Totals rise with population (rho = 0.83) but not significantly, and net sales per store are nearly the same in every county (about 3.6M to 3.9M KES).
Totals follow the number of stores, not population.

**Payment mix is the same everywhere:**

About 67% M-Pesa, 22% Cash, 10% Card by transactions and by net sales (Unknown 2%). Dry Foods, Household and Beverages are the top 3 categories for every channel, and all channels peak on Tuesday.

**Limitations**

- Each promotion covers a full calendar month, so there is no same-month control group; the promotion comparison cannot fully rule out seasonality.
- Only 22 holidays and 6 counties: small samples can hide modest effects, so "not significant" means no evidence, not proven no effect. Inflation covers 21 months(October 2026 has no inflation figure).
- Tuesday and Wednesday sell much more than other days, and some holidays fall in promotion months.
- The payment-by-category test treats basket lines as independent, which overstates significance; the effect size (V) is the more reliable number.
- The sales data is synthetic, so results describe this dataset only.
  

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

sales = pd.read_csv("../data/processed/sales_clean.csv", parse_dates=["transaction_date"])
sales["is_promo"] = sales["promo_id"] != "PROMO_NONE"

daily = (
    sales.groupby("transaction_date")
    .agg(net_sales=("net_sales_kes", "sum"),
         gross_profit=("gross_profit_kes", "sum"),
         transactions=("transaction_id", "nunique"),
         is_promo=("is_promo", "max"))
    .reset_index()
)
daily.groupby("is_promo")[["net_sales", "gross_profit", "transactions"]].mean().round(0)

,net_sales,gross_profit,transactions
is_promo,,,
False,58753.0,16028.0,88.0
True,50422.0,8046.0,88.0


In [2]:
def promo_test(metric):
    a = daily.loc[daily["is_promo"], metric]
    b = daily.loc[~daily["is_promo"], metric]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    diff = a.mean() - b.mean()
    va, vb = a.var(ddof=1)/len(a), b.var(ddof=1)/len(b)
    se = np.sqrt(va + vb)
    dof = (va + vb)**2 / (va**2/(len(a)-1) + vb**2/(len(b)-1))
    lo, hi = stats.t.interval(0.95, dof, loc=diff, scale=se)
    print(f"{metric}: promo {a.mean():,.0f} vs non-promo {b.mean():,.0f}")
    print(f"  difference {diff:,.0f}, 95% CI ({lo:,.0f}, {hi:,.0f}), t = {t:.2f}, p = {p:.4g}")

promo_test("net_sales")        # main test
promo_test("gross_profit")     # optional: did promos pay for their discounts?
promo_test("transactions")     # optional: did promos bring more customers?

net_sales: promo 50,422 vs non-promo 58,753
  difference -8,331, 95% CI (-10,200, -6,462), t = -8.76, p = 4.048e-17
gross_profit: promo 8,046 vs non-promo 16,028
  difference -7,982, 95% CI (-8,440, -7,524), t = -34.24, p = 3.382e-135
transactions: promo 88 vs non-promo 88
  difference -0, 95% CI (-3, 3), t = -0.03, p = 0.9751


In [3]:
p = "../data/processed/"
holidays = pd.read_csv(p + "holidays_clean.csv")
stores = pd.read_csv(p + "stores_clean.csv")
products = pd.read_csv(p + "products_clean.csv")
inflation = pd.read_csv(p + "inflation_clean.csv")
census = pd.read_csv(p + "census_clean.csv")

print(holidays, "\n")
print(stores, "\n")
print(products.head(), "\n")
print(inflation.head(), "\n")
print(census)

          Date        Day               Holiday_Name
0   2026-10-20    Tuesday               Mashujaa Day
1   2026-10-10   Saturday              Mazingira Day
2   2026-06-01     Monday               Madaraka Day
3   2026-05-27  Wednesday              Eid al-Adha *
4   2026-05-01     Friday                 Labour Day
..         ...        ...                        ...
93  2020-04-13     Monday              Easter Monday
94  2020-04-10     Friday                Good Friday
95  2020-03-20     Friday              Eid al-Fitr *
96  2020-02-11    Tuesday  Public Mourning — Death o
97  2020-01-01  Wednesday             New Year's Day

[98 rows x 3 columns] 

  store_id                     store_name       county      sub_county  \
0  STR_001  Nairobi Megastore - Westlands      Nairobi       Westlands   
1  STR_002          Nairobi Express - CBD      Nairobi         Starehe   
2  STR_003      Nairobi Local - Eastlands      Nairobi        Embakasi   
3  STR_004     Mombasa Coastal Superstore  

In [4]:
from scipy.stats import mannwhitneyu

holidays["Date"] = pd.to_datetime(holidays["Date"])
hol_dates = set(holidays.loc[(holidays["Date"] >= "2025-01-01") &
                             (holidays["Date"] <= "2026-10-31"), "Date"])
print(len(hol_dates), "holidays in the sales period")

d = daily.copy()
d["basket"] = d["net_sales"] / d["transactions"]

# the 3 days before each holiday (a holiday itself stays "holiday")
pre_dates = {h - pd.Timedelta(days=k) for h in hol_dates for k in (1, 2, 3)} - hol_dates

d["day_type"] = np.where(d["transaction_date"].isin(hol_dates), "holiday",
                np.where(d["transaction_date"].isin(pre_dates), "pre_holiday", "ordinary"))

print(d["day_type"].value_counts())
print(d.groupby("day_type")[["net_sales", "basket"]].mean().round(0))

def holiday_test(metric, group):
    a = d.loc[d["day_type"] == group, metric]
    b = d.loc[d["day_type"] == "ordinary", metric]
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    pct = (a.mean() / b.mean() - 1) * 100
    print(f"{metric}: {group} vs ordinary -> {pct:+.1f}% (n={len(a)} vs {len(b)}), U={u:.0f}, p={p:.4f}")

for m in ["net_sales", "basket"]:
    for g in ["holiday", "pre_holiday"]:
        holiday_test(m, g)

22 holidays in the sales period
day_type
ordinary       590
pre_holiday     57
holiday         22
Name: count, dtype: int64
             net_sales  basket
day_type                      
holiday        53808.0   628.0
ordinary       56168.0   639.0
pre_holiday    56286.0   623.0
net_sales: holiday vs ordinary -> -4.2% (n=22 vs 590), U=5989, p=0.5388
net_sales: pre_holiday vs ordinary -> +0.2% (n=57 vs 590), U=16910, p=0.9444
basket: holiday vs ordinary -> -1.7% (n=22 vs 590), U=5773, p=0.3789
basket: pre_holiday vs ordinary -> -2.5% (n=57 vs 590), U=14735, p=0.1228


In [5]:
from scipy.stats import spearmanr

monthly = (
    sales.assign(month_start=sales["transaction_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month_start")
    .agg(net_sales=("net_sales_kes", "sum"),
         transactions=("transaction_id", "nunique"),
         days=("transaction_date", "nunique"))
    .reset_index()
)
monthly["avg_daily_sales"] = monthly["net_sales"] / monthly["days"]
monthly["basket"] = monthly["net_sales"] / monthly["transactions"]

infl = inflation.copy()
infl["month_start"] = pd.to_datetime(infl["month_start"])
m = monthly.merge(infl[["month_start", "12-Month Inflation"]], on="month_start", how="inner")
print(len(m), "months used")

for col in ["net_sales", "avg_daily_sales", "basket"]:
    rho, p = spearmanr(m[col], m["12-Month Inflation"])
    print(f"{col}: rho = {rho:.2f}, p = {p:.3f}")

21 months used
net_sales: rho = -0.12, p = 0.618
avg_daily_sales: rho = -0.10, p = 0.679
basket: rho = -0.12, p = 0.602


In [6]:
# which month had no inflation figure?
print(sorted(set(monthly["month_start"]) - set(m["month_start"])))

cs = (
    sales.merge(stores[["store_id", "county"]], on="store_id", how="left")
    .groupby("county")
    .agg(net_sales=("net_sales_kes", "sum"), stores=("store_id", "nunique"))
    .reset_index()
)
cs = cs.merge(census[["County", "Total_Population19"]],
              left_on="county", right_on="County", how="left")
print(cs["Total_Population19"].isna().sum(), "counties without a census match")

cs["sales_per_1000"] = cs["net_sales"] / (cs["Total_Population19"] / 1000)
print(cs[["county", "stores", "Total_Population19", "net_sales", "sales_per_1000"]]
      .sort_values("net_sales", ascending=False).round(0).to_string(index=False))

rho, p = spearmanr(cs["Total_Population19"], cs["net_sales"])
print(f"Spearman (population vs total net sales): rho = {rho:.2f}, p = {p:.3f}, n = {len(cs)} counties")

[Timestamp('2026-10-01 00:00:00')]
0 counties without a census match
     county  stores  Total_Population19  net_sales  sales_per_1000
    Nairobi       3             4397073 11259289.0          2561.0
    Mombasa       2             1208333  7604004.0          6293.0
     Kiambu       2             2417735  7467443.0          3089.0
     Nakuru       1             2162202  3871652.0          1791.0
Uasin Gishu       1             1163186  3714230.0          3193.0
     Kisumu       1             1155574  3614539.0          3128.0
Spearman (population vs total net sales): rho = 0.83, p = 0.042, n = 6 counties


In [7]:
from itertools import permutations

x = cs["Total_Population19"].rank().values
y = cs["net_sales"].rank().values
obs = spearmanr(x, y)[0]
rhos = [spearmanr(x, list(p))[0] for p in permutations(y)]
p_exact = np.mean(np.abs(rhos) >= abs(obs) - 1e-9)
print(f"rho = {obs:.2f}, exact two-sided p = {p_exact:.3f}")

rho = 0.83, exact two-sided p = 0.058


In [8]:
print(sales["payment_method"].value_counts(dropna=False))

multi = (sales.groupby("transaction_id")["payment_method"].nunique() > 1).sum()
print(multi, "transactions with more than one payment method")

q1 = (
    sales
    .merge(products[["product_id", "category"]], on="product_id", how="left")
    .merge(stores[["store_id", "county"]], on="store_id", how="left")
)
print(q1[["category", "county"]].isna().sum())

tx = q1.drop_duplicates("transaction_id")
share = pd.DataFrame({
    "share_of_transactions": tx["payment_method"].value_counts(normalize=True),
    "share_of_net_sales": q1.groupby("payment_method")["net_sales_kes"].sum() / q1["net_sales_kes"].sum(),
}).round(3)
print(share)

payment_method
M-Pesa     75550
Cash       24393
Card       10927
Unknown     2298
Name: count, dtype: int64
0 transactions with more than one payment method
category    0
county      0
dtype: int64
                share_of_transactions  share_of_net_sales
payment_method                                           
Card                            0.097               0.095
Cash                            0.215               0.216
M-Pesa                          0.668               0.668
Unknown                         0.020               0.020


In [9]:
from scipy.stats import chi2_contingency

known = q1[q1["payment_method"] != "Unknown"]
tx_known = known.drop_duplicates("transaction_id")

def chi_cramer(df, col):
    ct = pd.crosstab(df["payment_method"], df[col])
    chi2, p, dof, exp = chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"payment x {col}: n={n:,}, chi2={chi2:.1f}, dof={dof}, p={p:.4g}, "
          f"Cramer's V={v:.3f}, smallest expected count={exp.min():.0f}")

chi_cramer(tx_known, "county")   # one row per transaction
chi_cramer(known, "category")    # one row per basket line

print((pd.crosstab(known["category"], known["payment_method"], normalize="index") * 100).round(1))
print((pd.crosstab(tx_known["county"], tx_known["payment_method"], normalize="index") * 100).round(1))

payment x county: n=57,733, chi2=13.6, dof=10, p=0.1931, Cramer's V=0.011, smallest expected count=560
payment x category: n=110,870, chi2=8.7, dof=12, p=0.7294, Cramer's V=0.006, smallest expected count=1447
payment_method  Card  Cash  M-Pesa
category                          
Bakery          10.1  21.6    68.2
Beverages       10.1  22.4    67.5
Dairy            9.7  21.8    68.4
Dry Foods        9.9  22.0    68.1
Fresh Produce    9.7  21.9    68.4
Household        9.7  21.9    68.4
Personal Care    9.6  22.4    68.1
payment_method  Card  Cash  M-Pesa
county                            
Kiambu           9.7  21.5    68.8
Kisumu           9.2  22.1    68.7
Mombasa          9.8  21.5    68.7
Nairobi         10.3  22.4    67.4
Nakuru          10.1  21.7    68.2
Uasin Gishu      9.7  22.4    67.9


In [10]:
# top 3 categories per payment channel, by net sales
cat_sales = known.groupby(["payment_method", "category"])["net_sales_kes"].sum().reset_index()
top3 = (cat_sales.sort_values(["payment_method", "net_sales_kes"], ascending=[True, False])
        .groupby("payment_method").head(3))
print(top3.round(0).to_string(index=False))

# average daily transactions by weekday, per payment channel
t = tx_known.copy()
t["date"] = t["transaction_date"].dt.normalize()
t["weekday"] = t["date"].dt.day_name()
per_day = t.groupby(["payment_method", "date", "weekday"]).size().reset_index(name="tx")
wd = per_day.groupby(["weekday", "payment_method"])["tx"].mean().unstack().round(1)
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(wd.reindex(order))
print("Peak day:", wd.idxmax().to_dict())

payment_method  category  net_sales_kes
          Card Dry Foods      1314412.0
          Card Household       598732.0
          Card Beverages       543193.0
          Cash Dry Foods      3049695.0
          Cash Household      1384227.0
          Cash Beverages      1164105.0
        M-Pesa Dry Foods      9204494.0
        M-Pesa Household      4336757.0
        M-Pesa Beverages      3574648.0
payment_method  Card  Cash  M-Pesa
weekday                           
Monday           8.8  19.9    60.8
Tuesday         10.9  23.5    72.5
Wednesday        9.8  20.9    66.7
Thursday         7.8  17.0    52.4
Friday           7.6  17.5    52.2
Saturday         7.6  16.8    53.9
Sunday           7.3  16.9    53.5
Peak day: {'Card': 'Tuesday', 'Cash': 'Tuesday', 'M-Pesa': 'Tuesday'}
